# Course 8: Mathematics and AI
## Session 12: AI for mathematical discovery

**Goal:** build a small evolve-and-evaluate loop that searches for programs producing large cap sets, with a trusted evaluator and a pluggable mutation step.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every Python cell without a NOT RUN label was run once on a CPU (November 2026). Cells labelled NOT RUN need a library, a GPU or a Lean installation. Lean code in this course was written carefully but was NOT compiled here: check it in Lean before you teach it.

### 1. The pattern
FunSearch and AlphaEvolve share one loop:
1. keep a population of programs,
2. ask a language model to modify a good one,
3. run the new program and score it with an evaluator that cannot be fooled,
4. keep it if it scores well.

The model supplies ideas. The evaluator supplies truth. Here the 'language model' is replaced by random edits to a small program, so the loop runs anywhere; plugging in a real model is the last cell.

### 2. The problem: cap sets
In the game SET, cards are vectors in `{0, 1, 2}^4` and three cards form a 'set' when they sum to zero mod 3 in every coordinate (equivalently, they lie on a line). A cap set is a collection with no such triple. The largest cap set in dimension 4 has 20 elements, and in dimension 5 it has 45. We work in dimension 5, where a random greedy order falls well short. FunSearch found new large cap sets in dimension 8 by evolving a *priority function* used by a greedy builder, and that is what we do here.

In [ ]:
import itertools, random, numpy as np
n = 5
points = np.array(list(itertools.product(range(3), repeat=n)))                 # all 243 points
code_of = lambda v: int(sum(int(c) * 3 ** (n - 1 - i) for i, c in enumerate(v)))
N_PTS = len(points)
# third[i][j] = index of the unique point that completes a line with points i and j
third = [[code_of((-points[i] - points[j]) % 3) for j in range(N_PTS)] for i in range(N_PTS)]

def is_cap_set(S):
    """The trusted evaluator. S is a list of point indices. No three different points may sum to zero mod 3."""
    chosen = set(S)
    if len(chosen) != len(S): return False
    return not any(third[a][b] in chosen and third[a][b] not in (a, b) for a, b in itertools.combinations(S, 2))

def greedy(priority):
    """Add points in order of priority (a list of 243 scores), skipping any point that would complete a line."""
    order = sorted(range(N_PTS), key=lambda i: -priority[i]); S, blocked = [], set()
    for v in order:
        if v in blocked: continue
        for a in S: blocked.add(third[a][v])
        S.append(v); blocked.add(v)
    return S

rng = random.Random(0)
random_sizes = [len(greedy([rng.random() for _ in range(N_PTS)])) for _ in range(200)]
print("greedy with a random order: mean size %.1f, best %d (the true maximum in dimension 5 is 45)" % (np.mean(random_sizes), max(random_sizes)))

### 3. Programs as priority functions
A 'program' here is a small formula over features of a point: how many coordinates are 0, 1 or 2, some pairwise sums, and so on. Its genome is a weight per feature. A language model would write arbitrary Python instead; the loop around it is the same.

In [ ]:
def features(v):
    c = [int((v == k).sum()) for k in range(3)]
    f = [c[0], c[1], c[2], c[0] * c[1], c[1] * c[2], c[0] * c[2], int(v.sum() % 3 == 0), int(v.sum() % 3 == 1)]
    f += [int((v[i] + v[j]) % 3 == 0) for i, j in itertools.combinations(range(n), 2)]            # pairwise sums
    f += [int((v[i] - v[j]) % 3 == 0) for i, j in itertools.combinations(range(n), 2)]            # pairwise differences
    return np.array(f, dtype=float)
PHI = np.stack([features(v) for v in points]); D = PHI.shape[1]

def evaluate(w):
    S = greedy((PHI @ w).tolist())
    assert is_cap_set(S)                              # never trust a score the evaluator has not verified
    return len(S)

def mutate(w, rng):
    w = w.copy(); kind = rng.random()
    if kind < 0.6: i = rng.randrange(D); w[i] += rng.gauss(0, 1)             # tweak one term
    elif kind < 0.8: i = rng.randrange(D); w[i] = 0.0                         # delete a term
    else: w += np.array([rng.gauss(0, 0.3) for _ in range(D)])                # small change everywhere
    return w

### 4. The evolutionary loop, with islands
Several separate populations ('islands') evolve on their own, which keeps different ideas alive. Now and then the worst island is restarted from the best one.

In [ ]:
def evolve(generations=3000, n_islands=4, island_size=10, seed=0):
    rng = random.Random(seed); np.random.seed(seed)
    islands = [[(evaluate(w), w) for w in [np.array([rng.gauss(0, 1) for _ in range(D)]) for _ in range(island_size)]] for _ in range(n_islands)]
    history = []
    for g in range(generations):
        isl = islands[g % n_islands]
        parent = max(rng.sample(isl, 3), key=lambda t: t[0])[1]                # tournament selection
        child = mutate(parent, rng); score = evaluate(child)
        worst = min(range(len(isl)), key=lambda i: isl[i][0])
        if score >= isl[worst][0]: isl[worst] = (score, child)                 # accept ties: lets the search drift across plateaus
        if (g + 1) % 300 == 0:                                                  # migration: restart the weakest island from the strongest
            best_i = max(range(n_islands), key=lambda i: max(t[0] for t in islands[i])); worst_i = min(range(n_islands), key=lambda i: max(t[0] for t in islands[i]))
            islands[worst_i] = list(islands[best_i])
        history.append(max(t[0] for isl_ in islands for t in isl_))
    best = max((t for isl_ in islands for t in isl_), key=lambda t: t[0])
    return best, history

(best_score, best_w), history = evolve()
print("best cap set size by generation:", {g: history[g - 1] for g in [1, 100, 500, 1000, 2000, 3000]})
print(f"evolved priority function builds a cap set of size {best_score} | random orders averaged {np.mean(random_sizes):.1f} with best {max(random_sizes)} | true maximum 45")
assert best_score >= np.mean(random_sizes) + 3

### 5. Verify independently, and read the result
The evaluator inside the loop already checked every candidate. For anything you would report, check once more with separate code, and look at the program that won.

In [ ]:
S = greedy((PHI @ best_w).tolist())
lines = sum(1 for a, b, c in itertools.combinations(S, 3) if not ((points[a] + points[b] + points[c]) % 3).any())   # a slower, separate check
print("size", len(S), "| lines found by the independent check:", lines)
assert lines == 0 and len(S) == best_score
names = ["#zeros", "#ones", "#twos", "#0*#1", "#1*#2", "#0*#2", "sum=0", "sum=1"] + [f"x{i}+x{j}=0" for i, j in itertools.combinations(range(n), 2)] + [f"x{i}=x{j}" for i, j in itertools.combinations(range(n), 2)]
top = np.argsort(-np.abs(best_w))[:5]
print("largest terms of the winning priority function:", [(names[i], round(float(best_w[i]), 2)) for i in top])

A discovery is the object (the cap set) together with a check anyone can rerun. The priority function is the explanation, and reading it is where a mathematician might spot a pattern worth proving. FunSearch's authors did exactly this with their dimension-8 construction.

### 6. What makes a problem suitable
- A fast, exact evaluator. If scoring needs human judgement, this loop does not apply.
- A score with gradations, so partial progress is visible (size of the set, not just 'is it a record').
- A compact program that generalizes: evolve the function that builds the object, not the object itself.
- An honest baseline: compare with random search under the same number of evaluations.

In [ ]:
rng = random.Random(1)
random_search = max(evaluate(np.array([rng.gauss(0, 1) for _ in range(D)])) for _ in range(3040))    # same number of evaluations as the loop
print(f"random search over priority functions, same budget: {random_search} | evolution: {best_score}")

Read this comparison honestly. When random search with the same budget matches the loop, as it can on a problem this small, the evolutionary step is not what found the answer: the greedy builder and the choice of features did the work. The published systems earn their results on search spaces where random search gets nowhere. Always run this baseline before claiming that a clever loop discovered something.

### 7. Put a language model in the loop (NOT RUN here)

In [ ]:
# NOT RUN: needs a code-capable language model (a small open one on a GPU, or an API).
PROMPT = """You are improving a Python function used to build large cap sets in {{0,1,2}}^{n}.
A greedy algorithm adds points in decreasing order of priority(v), skipping any point that completes a line.
Here are two earlier versions and their scores (larger is better):

{version_a}   # score {score_a}

{version_b}   # score {score_b}

Write a new function `priority(v)` (v is a tuple of {n} integers in 0..2) that you expect to score higher.
Return only the code."""

def llm_mutate(parents, llm):
    code = llm(PROMPT.format(n=n, version_a=parents[0][1], score_a=parents[0][0], version_b=parents[1][1], score_b=parents[1][0]))
    return code

def safe_evaluate(code, timeout=5):
    # Run untrusted code in a separate process with a time limit and no network or file access (for example a subprocess in a sandbox).
    # Inside: exec the code, build the set with `greedy`, verify with `is_cap_set`, return the size, or 0 on any error.
    raise NotImplementedError
# Loop: sample two parents from an island, call llm_mutate, score with safe_evaluate, insert if it is good. Log every program and score.
# Try n = 5 (maximum 45) and n = 6 (maximum 112). Report the best size, the number of model calls, and the random-search baseline.

### TODO
1. Run `evolve` with 5 seeds. Report the distribution of the best size, not the best run. Does any seed reach 45?
2. Remove the islands (one population of 40). Does the search get stuck more often?
3. Change the problem: find a large set of numbers in 1..60 with no three in arithmetic progression. What must change, and what stays the same?
4. Write half a page: where did the 'intelligence' in this loop come from, the mutation step or the evaluator?